In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.signal import convolve2d


class Downsampler(nn.Module):

    def __init__(self, factor, mode='lanczos'):

        super(Downsampler, self).__init__()

        self.factor = factor
        self.mode = mode

        if mode == 'lanczos':

            support = 2
            kernel_width = 4 * factor + 1

            self.kernel = self._generate_lanczos_kernel(
                factor,
                kernel_width,
                support
            )

            self.downsampler_ = self._create_downsampler(factor)

            pad = self._calculate_padding(factor)

            self.padding = nn.ReplicationPad2d(pad)

        elif mode == 'avg':

            self.downsampler_ = nn.AvgPool2d(
                kernel_size=factor,
                stride=factor
            )

        elif mode == 'bilinear':

                self.downsampler_ = lambda x: F.interpolate(
                    x,
                    scale_factor=1/factor,
                    mode='bilinear',
                    align_corners=False)

        else:

            raise ValueError(
                "mode must be "
                "'lanczos', "
                "'avg', "
                "'bilinear'"
            )

    def _generate_lanczos_kernel(self, factor, kernel_width, support):

        kernel = np.zeros((kernel_width, kernel_width), dtype=np.float32)

        center = (kernel_width - 1) / 2.0

        for i in range(kernel_width):
            for j in range(kernel_width):

                di = abs(i - center) / factor
                dj = abs(j - center) / factor

                kernel[i, j] = (
                    self._lanczos_value(di, support)
                    * self._lanczos_value(dj, support)
                )

        kernel /= kernel.sum()

        kernel = 0.5 * (kernel + kernel[::-1, ::-1])

        return kernel

    @staticmethod
    def _lanczos_value(
        distance,
        support
    ):

        if distance == 0:
            return 1.0

        pi_d = np.pi * distance

        return (
            support
            *
            np.sin(pi_d)
            *
            np.sin(
                pi_d / support
            )
            /
            (
                pi_d * pi_d
            )
        )


    def _create_downsampler(
        self,
        factor
    ):

        kernel_size = (
            self.kernel.shape[0]
        )

        downsampler = nn.Conv2d(
            1,
            1,
            kernel_size=kernel_size,
            stride=factor,
            padding=0,
            bias=False
        )

        downsampler.weight.data.zero_()

        kernel_torch = (
            torch.from_numpy(
                self.kernel
            ).float()
        )

        downsampler.weight.data[
            0, 0
        ] = kernel_torch

        return downsampler

    def _calculate_padding(
        self,
        factor
    ):

        kernel_size = (
            self.kernel.shape[0]
        )

        if kernel_size % 2 == 1:

            pad = (
                kernel_size - 1
            ) // 2

        else:

            pad = (
                kernel_size - factor
            ) // 2

        return pad

    def _center_crop(
        self,
        kernel,
        target_size
    ):

        if isinstance(
            target_size,
            int
        ):
            target_size = (
                target_size,
                target_size
            )

        h, w = kernel.shape

        th, tw = target_size

        i = (h - th) // 2

        j = (w - tw) // 2

        return kernel[
            i:i + th,
            j:j + tw
        ]

    def forward(
        self,
        input
    ):

        if self.mode == 'lanczos':

            x = self.padding(
                input
            )

            return self.downsampler_(
                x
            )

        elif self.mode == 'avg':

            return self.downsampler_(
                input
            )

        elif self.mode == 'bilinear':

            return F.interpolate(
                input,
                scale_factor=(
                    1.0 /
                    self.factor
                ),
                mode='bilinear',
                align_corners=False,
                antialias=True
            )